# Build Graph NPZ From Zipped PCAPs On Kaggle

Use this notebook after pushing the latest repo code to GitHub. Kaggle flow:

```text
clone repo -> pip install repo -> run notebooks/build_graph_npz_from_zipped_pcaps_kaggle.py -> download graph_npz_artifact.zip
```

The Python driver in the repo does the real work. This notebook is only the Kaggle wrapper.

In [ ]:
from __future__ import annotations

import shutil
import subprocess
import sys
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/LeThanhPhat-ATTT2023/Do-an-chuyen-nganh_NT114.git"
GITHUB_BRANCH = ""  # set branch/commit after pushing latest code; empty = default branch
FORCE_RECLONE = False

REPO_DIR = Path("/kaggle/working/Do-an-chuyen-nganh_NT114")
WORK_DIR = Path("/kaggle/working/nt114_npz_work")
OUTPUT_ZIP = Path("/kaggle/working/graph_npz_artifact.zip")

RESET_WORK_DIR = False
PAYLOAD_EXTRACT_WORKERS = 0  # 0 = auto min(num_pcap_files, CPU cores)
LOG_EVERY_PACKETS = 100_000
WRITE_BATCH_SIZE = 100_000
MAX_PACKETS_PER_FILE = None  # set int for smoke test only

TEACHER_MODEL_NAME = "ehsanaghaei/SecureBERT"
TEACHER_BATCH_SIZE = 32
STUDENT_EPOCHS = 30
STUDENT_BATCH_SIZE = 256
STUDENT_NUM_WORKERS = 2
STUDENT_EMB_BATCH_SIZE = 1024
MITRE_BATCH_SIZE = 64
DEVICE = "auto"

SIMILARITY_THRESHOLD = 0.82
PACKET_TOP_K = 5
FLOW_TOP_K = 5


In [ ]:
def run(cmd: list[str], cwd: Path | None = None) -> None:
    print("\n$", " ".join(str(x) for x in cmd), flush=True)
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


if FORCE_RECLONE and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run(["git", "clone", GITHUB_REPO_URL, str(REPO_DIR)])

if GITHUB_BRANCH:
    run(["git", "fetch", "--all"], cwd=REPO_DIR)
    run(["git", "checkout", GITHUB_BRANCH], cwd=REPO_DIR)

run([sys.executable, "-m", "pip", "install", "-q", "scapy>=2.5.0", "transformers>=4.40", "sentencepiece", "accelerate"])
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])

script = REPO_DIR / "notebooks" / "build_graph_npz_from_zipped_pcaps_kaggle.py"
if not script.exists():
    raise FileNotFoundError(f"Missing repo driver script: {script}")
print("Driver:", script)


In [ ]:
script = REPO_DIR / "notebooks" / "build_graph_npz_from_zipped_pcaps_kaggle.py"
cmd = [
    sys.executable,
    "-u",
    str(script),
    "--input-root",
    "/kaggle/input",
    "--work-dir",
    str(WORK_DIR),
    "--output-zip",
    str(OUTPUT_ZIP),
    "--payload-extract-workers",
    str(PAYLOAD_EXTRACT_WORKERS),
    "--log-every-packets",
    str(LOG_EVERY_PACKETS),
    "--write-batch-size",
    str(WRITE_BATCH_SIZE),
    "--teacher-model-name",
    TEACHER_MODEL_NAME,
    "--teacher-batch-size",
    str(TEACHER_BATCH_SIZE),
    "--student-epochs",
    str(STUDENT_EPOCHS),
    "--student-batch-size",
    str(STUDENT_BATCH_SIZE),
    "--student-num-workers",
    str(STUDENT_NUM_WORKERS),
    "--student-emb-batch-size",
    str(STUDENT_EMB_BATCH_SIZE),
    "--mitre-batch-size",
    str(MITRE_BATCH_SIZE),
    "--device",
    DEVICE,
    "--similarity-threshold",
    str(SIMILARITY_THRESHOLD),
    "--packet-top-k",
    str(PACKET_TOP_K),
    "--flow-top-k",
    str(FLOW_TOP_K),
]
if RESET_WORK_DIR:
    cmd.append("--reset-work-dir")
if MAX_PACKETS_PER_FILE is not None:
    cmd += ["--max-packets-per-file", str(MAX_PACKETS_PER_FILE)]

run(cmd, cwd=REPO_DIR)


In [ ]:
if not OUTPUT_ZIP.exists():
    raise FileNotFoundError(OUTPUT_ZIP)

print("Download:", OUTPUT_ZIP)
print("Size GB:", round(OUTPUT_ZIP.stat().st_size / 1024**3, 3))
print("Working outputs:")
for path in sorted(WORK_DIR.glob("data/processed/graph_artifact_3tier_t082_k5*")):
    print(" -", path, round(path.stat().st_size / 1024**3, 3), "GB")
